# JupyterLite で学ぶ パネルデータ計量経済学 入門チュートリアル（固定効果・変量効果・操作変数）

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
**パネルデータ**（同じ個体を複数期間にわたって観測したデータ）を使った計量経済学の基本手法を、
**statsmodels** で実践しながら学ぶためのチュートリアルです。

## 対象者
- 単回帰・重回帰（OLS）の基本を学んだ方
- 「固定効果」「変量効果」「操作変数」「差の差」という言葉を聞いたことはあるが、自分で推定したことがない方
- 卒業論文などで企業・都道府県・個人のパネルデータを分析したい方

## このチュートリアルで学ぶこと
0. 環境準備（JupyterLite 用）
1. パネルデータとは
2. 合成パネルデータの作成
3. プーリング OLS とその問題点
4. 固定効果モデル（within 変換・ダミー変数・時間効果・クラスター標準誤差）
5. 変量効果モデルとハウスマン検定の考え方
6. 操作変数法（2SLS）と弱操作変数
7. 差の差（DID）
8. 結果表の作成
9. まとめと総合演習

## JupyterLite での注意
パネルデータ専用ライブラリ **linearmodels** は、現在の JupyterLite（Pyodide）環境では動作しません
（純 Python の wheel が配布されておらず、古い版は pandas 3 と非互換）。そこでこのノートブックでは、
Pyodide に同梱されている **statsmodels** だけで同じ推定を行います。ローカル環境で linearmodels を使う場合の
書き方は、各章の末尾に「ローカル環境で linearmodels を使う場合」として示します。

---
## 0. 環境準備（JupyterLite 用）

In [ ]:
# JupyterLite 用のパッケージインストール（初回は数十秒かかることがあります）
try:
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "scipy", "statsmodels", "japanize-matplotlib-jlite"])
    print("piplite でのインストールが完了しました")
except ImportError:
    print("piplite がない環境（ローカルの Jupyter）なのでスキップしました")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語フォント（pyplot の後に import）
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from statsmodels.sandbox.regression.gmm import IV2SLS
from scipy import stats

pd.set_option("display.width", 120)
print("statsmodels バージョン:", sm.__version__)
print("pandas バージョン     :", pd.__version__)

---
## 1. パネルデータとは

| データの種類 | 例 | 添え字 |
|---|---|---|
| クロスセクション | 2023 年の 47 都道府県の失業率 | $i$ |
| 時系列 | 愛知県の 1990〜2023 年の失業率 | $t$ |
| **パネルデータ** | 47 都道府県 × 1990〜2023 年の失業率 | $i, t$ |

パネルデータの基本モデルは次のように書けます。

$$
y_{it} = \beta x_{it} + \alpha_i + \lambda_t + \varepsilon_{it}
$$

- $\alpha_i$：**個体効果**（企業の経営能力、県の風土など、観測できないが時間を通じて一定の要因）
- $\lambda_t$：**時間効果**（景気変動など、全個体に共通する年ごとの要因）
- $\varepsilon_{it}$：誤差項

パネルデータの最大の利点は、**観測できない個体効果 $\alpha_i$ をコントロールできる** ことです。
$\alpha_i$ が説明変数 $x_{it}$ と相関していると、普通の OLS は偏った（バイアスのある）推定値を返します。
この問題を、同じ個体を何度も観測していることを利用して解決するのが固定効果モデルです。

---
## 2. 合成パネルデータの作成

企業 60 社 × 8 年（2016〜2023 年）の架空データを作ります。**真のモデル** は次のとおりです。

$$
\text{sales}_{it} = 1.0 + 0.5\,\text{ad}_{it} + 0.3\,\text{size}_{it} + \alpha_i + \lambda_t + \varepsilon_{it}
$$

- `sales`：売上（対数）、`ad`：広告費（対数）、`size`：企業規模（対数従業員数）
- 個体効果 $\alpha_i$ は「経営能力」のようなもので、**広告費と正の相関** を持たせてあります
  （能力の高い経営者ほど広告にも積極的、という状況）。これが OLS のバイアスの原因になります。
- 広告費の真の効果は **0.5** です。各手法がこの値をどれだけ正しく推定できるかを見ていきます。

In [ ]:
rng = np.random.default_rng(42)
n_firms, n_years = 60, 8
years = np.arange(2016, 2016 + n_years)

firm = np.repeat(np.arange(1, n_firms + 1), n_years)
year = np.tile(years, n_firms)

alpha = rng.normal(0, 1.0, n_firms)                 # 個体効果（経営能力）
lam = np.array([0, 0.1, 0.2, 0.15, -0.3, -0.1, 0.2, 0.3])   # 時間効果（景気）

ad = 2.0 + 0.8 * alpha[firm - 1] + rng.normal(0, 1.0, len(firm))   # 広告費は個体効果と相関
size = 4.0 + rng.normal(0, 0.5, len(firm))
eps = rng.normal(0, 0.5, len(firm))

sales = 1.0 + 0.5 * ad + 0.3 * size + alpha[firm - 1] + lam[year - 2016] + eps

panel = pd.DataFrame({"firm": firm, "year": year, "sales": sales, "ad": ad, "size": size})
print(panel.head(10))
print("\n観測数:", len(panel), "（企業", n_firms, "社 ×", n_years, "年）")

In [ ]:
print(panel.describe().round(3))

すべての企業が同じ年数だけ観測されているパネルを **バランスト・パネル**、欠けがあるものを **アンバランスト・パネル** と呼びます。
分析の前に、企業ごとの観測数を確認しておきましょう。

In [ ]:
obs_per_firm = panel.groupby("firm")["year"].count()
print("企業ごとの観測数の分布:")
print(obs_per_firm.value_counts())
print("バランスト・パネル:", obs_per_firm.nunique() == 1)

### ロング形式とワイド形式

上のように「1 行 = 1 企業 × 1 年」の形を **ロング形式** と呼び、回帰分析にはこの形を使います。
表計算ソフトでよく見る「行 = 企業、列 = 年」の **ワイド形式** には `pivot()` で変換できます。

In [ ]:
wide = panel.pivot(index="firm", columns="year", values="sales")
print(wide.head().round(2))

# ワイド形式 → ロング形式に戻す
long_again = wide.reset_index().melt(id_vars="firm", var_name="year", value_name="sales")
print("\nロング形式に戻した行数:", len(long_again))

In [ ]:
# いくつかの企業の売上の推移を描く
plt.figure(figsize=(8, 4))
for f in [1, 2, 3, 4, 5]:
    sub = panel[panel["firm"] == f]
    plt.plot(sub["year"], sub["sales"], marker="o", label=f"企業 {f}")
plt.xlabel("年")
plt.ylabel("売上（対数）")
plt.title("企業ごとの売上の推移")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## 3. プーリング OLS とその問題点

パネル構造を無視して、全 480 行をひとまとめにして OLS を行うのが **プーリング OLS** です。

In [ ]:
pooled = smf.ols("sales ~ ad + size", data=panel).fit()
print(pooled.summary().tables[1])
print("\n広告費の推定値:", round(pooled.params["ad"], 3), "（真の値 0.5）")

広告費の係数が真の値 0.5 より **大きく推定** されています。これは、能力の高い企業（$\alpha_i$ が大きい）ほど
広告費も売上も大きいため、「広告費の効果」の中に「経営能力の効果」が混ざってしまうからです（**欠落変数バイアス**）。

散布図で確かめてみましょう。企業ごとに色を分けると、「企業間の差（between）」が「企業内の変動（within）」より
急な傾きを作っていることが分かります。

In [ ]:
plt.figure(figsize=(7, 5))
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for i, f in enumerate([1, 2, 3, 4, 5, 6]):
    sub = panel[panel["firm"] == f]
    plt.scatter(sub["ad"], sub["sales"], color=colors[i], label=f"企業 {f}")
    # 企業ごとの回帰直線（within の傾き）
    b = np.polyfit(sub["ad"], sub["sales"], 1)
    xs = np.linspace(sub["ad"].min(), sub["ad"].max(), 10)
    plt.plot(xs, np.polyval(b, xs), color=colors[i], alpha=0.7)
# 全体の回帰直線（プーリング）
b_all = np.polyfit(panel["ad"], panel["sales"], 1)
xs = np.linspace(panel["ad"].min(), panel["ad"].max(), 10)
plt.plot(xs, np.polyval(b_all, xs), color="black", linestyle="--", linewidth=2, label="プーリング OLS")
plt.xlabel("広告費（対数）")
plt.ylabel("売上（対数）")
plt.title("企業内の傾き（色つき）とプーリングの傾き（黒破線）")
plt.legend(fontsize=8)
plt.show()

### 企業ごとの平均で回帰する（between 推定）

反対に、企業ごとの **平均値だけ** を使って回帰すると（**between 推定**）、企業間の差だけを見ることになり、
個体効果によるバイアスがもっとも強く出ます。

In [ ]:
between = panel.groupby("firm")[["sales", "ad", "size"]].mean()
between_res = smf.ols("sales ~ ad + size", data=between).fit()
print("between 推定の広告費の係数:", round(between_res.params["ad"], 3), "（真の値 0.5）")

---
## 4. 固定効果モデル

### 4.1 within 変換を手で行う

固定効果モデルの考え方はシンプルです。各変数から **企業ごとの平均を引く**（within 変換）と、
時間を通じて一定の $\alpha_i$ が消えます。

$$
y_{it} - \bar{y}_i = \beta (x_{it} - \bar{x}_i) + (\varepsilon_{it} - \bar{\varepsilon}_i)
$$

この変換後のデータに OLS をかければ、$\alpha_i$ の影響を受けない推定値が得られます。

In [ ]:
within = panel.copy()
for col in ["sales", "ad", "size"]:
    within[col + "_w"] = panel[col] - panel.groupby("firm")[col].transform("mean")

fe_manual = sm.OLS(within["sales_w"], within[["ad_w", "size_w"]]).fit()
print(fe_manual.params.round(3))
print("\n広告費の推定値:", round(fe_manual.params["ad_w"], 3), "（真の値 0.5）")

### 4.2 ダミー変数で推定する（LSDV）

企業ごとのダミー変数（0/1 の列）を入れて OLS を行っても、まったく同じ係数が得られます
（**最小二乗ダミー変数法, LSDV**）。statsmodels の式では `C(firm)` と書くだけです。
企業数が多いと係数の表が長くなりますが、`params["ad"]` のように必要な係数だけ取り出せます。

In [ ]:
fe_lsdv = smf.ols("sales ~ ad + size + C(firm)", data=panel).fit()
print("広告費の推定値（LSDV）  :", round(fe_lsdv.params["ad"], 3))
print("広告費の推定値（between）:", round(between_res.params["ad"], 3))
print("広告費の推定値（within）:", round(fe_manual.params["ad_w"], 3))
print("標準誤差（LSDV）        :", round(fe_lsdv.bse["ad"], 4))
print("推定した係数の数:", len(fe_lsdv.params), "（定数項 + ad + size + 企業ダミー 59 個）")

within 変換を手で行った場合、自由度の計算がずれるので **標準誤差はわずかに過小** になります。
実務では LSDV（`C(firm)`）を使うか、専用ライブラリ（linearmodels の `PanelOLS`）を使うのが確実です。

### 4.3 推定された固定効果を眺める

LSDV の企業ダミーの係数は、基準企業（企業 1）に対する各企業の固定効果です。
データを作るときに使った $\alpha_i$ と比べてみましょう。

In [ ]:
fe_coef = fe_lsdv.params.filter(like="C(firm)")
est_alpha = np.r_[0.0, fe_coef.values]                 # 企業 1 を 0 とした相対値
true_alpha_rel = alpha - alpha[0]                      # 真の値も企業 1 を基準に

plt.figure(figsize=(5, 5))
plt.scatter(true_alpha_rel, est_alpha)
lim = [min(true_alpha_rel.min(), est_alpha.min()) - 0.3, max(true_alpha_rel.max(), est_alpha.max()) + 0.3]
plt.plot(lim, lim, color="red", linestyle="--", label="45 度線")
plt.xlabel("真の個体効果（企業 1 基準）")
plt.ylabel("推定された固定効果")
plt.title("固定効果の推定値と真の値")
plt.legend()
plt.show()
print("相関係数:", round(np.corrcoef(true_alpha_rel, est_alpha)[0, 1], 3))

### 4.4 時間効果も入れる（二方向固定効果）

景気などの年ごとの共通ショック $\lambda_t$ も、年ダミー `C(year)` を加えれば取り除けます。
企業と年の両方の固定効果を入れたモデルを **二方向固定効果（two-way FE）モデル** と呼びます。

In [ ]:
fe_twoway = smf.ols("sales ~ ad + size + C(firm) + C(year)", data=panel).fit()
print("広告費の推定値（二方向 FE）:", round(fe_twoway.params["ad"], 3))
print("\n年効果の推定値（2016 年基準）:")
print(fe_twoway.params.filter(like="C(year)").round(3))
print("\n真の時間効果（2016 年基準）:", (lam - lam[0]).round(3))

### 4.5 一階差分モデル

個体効果を消すもう 1 つの方法が **一階差分（first difference）** です。各企業について前年からの変化
$\Delta y_{it} = y_{it} - y_{i,t-1}$ をとると、時間を通じて一定の $\alpha_i$ は消えます。
期間が 2 期だけなら固定効果と同じ推定値になり、期間が長いときは誤差項の性質によってどちらが効率的かが変わります。

In [ ]:
diffed = panel.sort_values(["firm", "year"]).copy()
for col in ["sales", "ad", "size"]:
    diffed["d_" + col] = diffed.groupby("firm")[col].diff()
diffed = diffed.dropna()
fd = smf.ols("d_sales ~ d_ad + d_size", data=diffed).fit()
print("一階差分モデルの広告費の係数:", round(fd.params["d_ad"], 3), "（真の値 0.5）")
print("使った観測数:", int(fd.nobs), "（各企業の最初の年が落ちる）")

### 4.6 クラスター頑健標準誤差

同じ企業の誤差項は年をまたいで相関している（系列相関）ことが普通です。これを無視すると標準誤差が過小になり、
本当は有意でない係数を有意と判定してしまいます。パネルデータでは **企業（個体）でクラスター化した標準誤差** を
使うのが標準的な作法です。`fit(cov_type="cluster", cov_kwds={"groups": ...})` で指定します。

In [ ]:
fe_cluster = smf.ols("sales ~ ad + size + C(firm) + C(year)", data=panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["firm"]}
)
se_table = pd.DataFrame({
    "係数": [fe_twoway.params["ad"], fe_cluster.params["ad"]],
    "標準誤差": [fe_twoway.bse["ad"], fe_cluster.bse["ad"]],
    "t 値": [fe_twoway.tvalues["ad"], fe_cluster.tvalues["ad"]],
}, index=["通常の標準誤差", "クラスター標準誤差"])
print(se_table.round(4))

係数は同じで、標準誤差だけが変わることを確認してください（この合成データでは系列相関を入れていないので差は小さいですが、
実データではクラスター標準誤差の方が大きくなるのが普通です）。

> **ローカル環境で linearmodels を使う場合**
> ```python
> from linearmodels.panel import PanelOLS
> data = panel.set_index(["firm", "year"])
> fe = PanelOLS(data["sales"], data[["ad", "size"]], entity_effects=True, time_effects=True)
> print(fe.fit(cov_type="clustered", cluster_entity=True))
> ```

### 練習問題 1

1. 企業規模 `size` を説明変数から外し、`sales ~ ad + C(firm)` の固定効果モデルを推定して、広告費の係数を確認してください。
2. within 変換を手で行う方法で、`year` ごとの平均も引く「二方向 within 変換」を実装し、`ad` の係数が 4.4 の二方向 FE と一致することを確かめてください（ヒント：$x_{it} - \bar{x}_i - \bar{x}_t + \bar{x}$）。
3. 4.6 のクラスター標準誤差を使ったモデルについて、広告費の係数の 95% 信頼区間を `conf_int()` で表示してください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
m1 = smf.ols("sales ~ ad + C(firm)", data=panel).fit()
print("ad の係数:", round(m1.params["ad"], 3))

# 2
tw = panel.copy()
for col in ["sales", "ad", "size"]:
    tw[col + "_w2"] = (
        panel[col]
        - panel.groupby("firm")[col].transform("mean")
        - panel.groupby("year")[col].transform("mean")
        + panel[col].mean()
    )
m2 = sm.OLS(tw["sales_w2"], tw[["ad_w2", "size_w2"]]).fit()
print("二方向 within の ad:", round(m2.params["ad_w2"], 3), " 二方向 FE の ad:", round(fe_twoway.params["ad"], 3))

# 3
print(fe_cluster.conf_int().loc["ad"].round(3))
```

</details>

---
## 5. 変量効果モデルとハウスマン検定の考え方

### 5.1 変量効果モデル

**変量効果（random effects, RE）モデル** は、個体効果 $\alpha_i$ を「説明変数と無相関なランダムな切片」とみなして推定します。
$\alpha_i$ を推定せずに済むので効率的（標準誤差が小さい）ですが、**$\alpha_i$ と説明変数が相関していると偏ります**。

statsmodels では、線形混合モデル `mixedlm` で「企業ごとにランダムな切片」を指定すると変量効果モデルになります。

In [ ]:
re_model = smf.mixedlm("sales ~ ad + size", data=panel, groups=panel["firm"]).fit()
print(re_model.summary().tables[1])
print("\n広告費の推定値（RE）:", round(re_model.params["ad"], 3), "（真の値 0.5）")

このデータでは個体効果と広告費が相関しているので、変量効果モデルはプーリング OLS ほどではないものの上方に偏ります。
逆に、**個体効果と説明変数が無相関なデータ** を作って比べてみましょう。

In [ ]:
rng_re = np.random.default_rng(1)
alpha2 = rng_re.normal(0, 1.0, n_firms)
ad2 = 2.0 + rng_re.normal(0, 1.0, len(firm))          # 個体効果と無相関な広告費
sales2 = 1.0 + 0.5 * ad2 + 0.3 * size + alpha2[firm - 1] + lam[year - 2016] + rng_re.normal(0, 0.5, len(firm))
panel2 = pd.DataFrame({"firm": firm, "year": year, "sales": sales2, "ad": ad2, "size": size})

fe2 = smf.ols("sales ~ ad + size + C(firm)", data=panel2).fit()
re2 = smf.mixedlm("sales ~ ad + size", data=panel2, groups=panel2["firm"]).fit()
compare = pd.DataFrame({
    "係数": [fe2.params["ad"], re2.params["ad"]],
    "標準誤差": [fe2.bse["ad"], re2.bse["ad"]],
}, index=["固定効果", "変量効果"])
print(compare.round(4))

個体効果が説明変数と無相関なら、固定効果も変量効果も真の値 0.5 に近く、**変量効果の方が標準誤差が小さい**（効率的）ことが分かります。

### 5.2 ハウスマン検定の考え方

「固定効果と変量効果のどちらを使うべきか」を判断するのが **ハウスマン検定** です。

- 帰無仮説：個体効果と説明変数は無相関（→ 変量効果でよい）
- 両方の推定値が大きく違えば帰無仮説を棄却し、固定効果を選ぶ

検定統計量は（1 変数の場合）次のように計算できます。

$$
H = \frac{(\hat\beta_{FE} - \hat\beta_{RE})^2}{\text{Var}(\hat\beta_{FE}) - \text{Var}(\hat\beta_{RE})} \sim \chi^2(1)
$$

In [ ]:
def hausman_1var(fe_res, re_res, name):
    """1 変数の簡易ハウスマン統計量。分散の差が負のときは判定不能として NaN を返す。"""
    diff = fe_res.params[name] - re_res.params[name]
    var_diff = fe_res.bse[name] ** 2 - re_res.bse[name] ** 2
    if var_diff <= 0:
        return np.nan, np.nan
    h_stat = diff ** 2 / var_diff
    p_value = 1 - stats.chi2.cdf(h_stat, df=1)
    return h_stat, p_value

for label, fe_res, re_res in [("個体効果と相関あり", fe_lsdv, re_model), ("個体効果と相関なし", fe2, re2)]:
    h, p = hausman_1var(fe_res, re_res, "ad")
    print(f"{label}: FE = {fe_res.params['ad']:.3f}, RE = {re_res.params['ad']:.3f}", end="  ")
    if np.isnan(h):
        print("→ 分散の差が負で判定不能（下の Mundlak 検定を使う）")
    else:
        print(f"→ H = {h:.2f}, p = {p:.4f} → {'固定効果を選ぶ' if p < 0.05 else '変量効果でもよい'}")

簡易版のハウスマン統計量は、有限標本では「分散の差」が負になって計算できないことがよくあります（上の 1 つ目がその例です）。
実務でよく使われる頑健な代替が **Mundlak 検定** です。変量効果モデルに **説明変数の個体平均**（$\bar{x}_i$）を加え、
その係数が有意なら「個体効果は説明変数と相関している → 固定効果を選ぶ」と判断します。

In [ ]:
def mundlak_test(data):
    d = data.copy()
    d["ad_mean"] = d.groupby("firm")["ad"].transform("mean")
    d["size_mean"] = d.groupby("firm")["size"].transform("mean")
    res = smf.mixedlm("sales ~ ad + size + ad_mean + size_mean", data=d, groups=d["firm"]).fit()
    return res.params["ad_mean"], res.pvalues["ad_mean"]

for label, data in [("個体効果と相関あり", panel), ("個体効果と相関なし", panel2)]:
    coef, p = mundlak_test(data)
    print(f"{label}: ad_mean の係数 = {coef:.3f}, p = {p:.4f} → {'固定効果を選ぶ' if p < 0.05 else '変量効果でもよい'}")

> **ローカル環境で linearmodels を使う場合**
> ```python
> from linearmodels.panel import RandomEffects
> re = RandomEffects(data["sales"], sm.add_constant(data[["ad", "size"]])).fit()
> ```

### 練習問題 2

1. `panel2`（個体効果と無相関なデータ）に対してプーリング OLS を行い、固定効果・変量効果と係数を比べてください。
2. `panel`（相関ありのデータ）について、`size` の係数でも簡易ハウスマン統計量を計算してください（`size` は個体効果と無相関に作ってあります。分散の差が負なら NaN になります）。
3. `mixedlm` の結果から、企業間のばらつき（`Group Var`）を表示してください（ヒント：`re_model.cov_re`）。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
pooled2 = smf.ols("sales ~ ad + size", data=panel2).fit()
print("プーリング:", round(pooled2.params["ad"], 3), " FE:", round(fe2.params["ad"], 3), " RE:", round(re2.params["ad"], 3))

# 2
h, p = hausman_1var(fe_lsdv, re_model, "size")
print(f"size: H = {h:.2f}, p = {p:.4f}")

# 3
print(re_model.cov_re)
```

</details>

---
## 6. 操作変数法（2SLS）と弱操作変数

### 6.1 内生性の問題

固定効果で取り除けるのは「時間を通じて一定の」要因だけです。次のような場合、説明変数 $x$ と誤差項が相関し（**内生性**）、
OLS も固定効果も偏ります。

- 逆の因果（売上が増えたから広告費を増やした）
- 時間とともに変わる欠落変数
- 説明変数の測定誤差

古典的な例は **教育の収益率** です。賃金を教育年数に回帰すると、観測できない「能力」が教育年数と賃金の両方を高めるため、
教育の効果が過大に推定されます。

### 6.2 操作変数（IV）

**操作変数** $z$ は、次の 2 条件を満たす変数です。

1. **関連性**：$z$ は $x$ と相関する（$z$ が教育年数を動かす）
2. **外生性**：$z$ は誤差項（能力）と無相関で、$x$ を通じてしか $y$ に影響しない

ここでは「自宅の近くに大学があったか（`near_college`）」を操作変数にした合成データを作ります。

In [ ]:
rng_iv = np.random.default_rng(10)
n_people = 1000
ability = rng_iv.normal(0, 1, n_people)                        # 観測できない能力
near_college = rng_iv.integers(0, 2, n_people)                 # 操作変数（0/1）
educ = 12 + 2.0 * near_college + 1.0 * ability + rng_iv.normal(0, 1.5, n_people)   # 教育年数
log_wage = 1.5 + 0.08 * educ + 0.3 * ability + rng_iv.normal(0, 0.3, n_people)      # 真の収益率 0.08

iv_data = pd.DataFrame({"log_wage": log_wage, "educ": educ, "near_college": near_college})
print(iv_data.head())
print("\n真の教育収益率: 0.08")

In [ ]:
ols_iv = smf.ols("log_wage ~ educ", data=iv_data).fit()
print("OLS の推定値:", round(ols_iv.params["educ"], 4), "（能力バイアスで過大）")

### 6.3 2 段階最小二乗法（2SLS）を手で行う

1. **第 1 段階**：$x$ を $z$ に回帰し、予測値 $\hat{x}$ を作る（$x$ のうち $z$ で説明できる「外生的な部分」だけを取り出す）
2. **第 2 段階**：$y$ を $\hat{x}$ に回帰する

In [ ]:
# 第 1 段階
first_stage = smf.ols("educ ~ near_college", data=iv_data).fit()
iv_data["educ_hat"] = first_stage.fittedvalues
print("第 1 段階：near_college の係数 =", round(first_stage.params["near_college"], 3))

# 第 2 段階
second_stage = smf.ols("log_wage ~ educ_hat", data=iv_data).fit()
print("第 2 段階：educ_hat の係数    =", round(second_stage.params["educ_hat"], 4), "（真の値 0.08）")

手で 2 段階に分けると係数は正しく求まりますが、**第 2 段階の標準誤差は正しくありません**（予測値を使っていることを考慮していないため）。
正しい標準誤差を得るには、専用の関数を使います。

### 6.4 statsmodels の IV2SLS

In [ ]:
y_iv = iv_data["log_wage"]
X_iv = sm.add_constant(iv_data[["educ"]])            # 説明変数（内生変数を含む）
Z_iv = sm.add_constant(iv_data[["near_college"]])    # 操作変数（外生変数 + 操作変数）

iv_res = IV2SLS(y_iv, X_iv, instrument=Z_iv).fit()
print(iv_res.summary().tables[1])
print("\n2SLS の推定値:", round(iv_res.params["educ"], 4), "（真の値 0.08）")

### 6.5 弱操作変数

操作変数と内生変数の相関が弱いと（**弱操作変数**）、2SLS の推定値は不安定で、OLS と同じ方向に偏ります。
判断の目安は **第 1 段階の F 統計量が 10 以上** かどうかです。

In [ ]:
print("第 1 段階の F 統計量:", round(first_stage.fvalue, 1), "→", "十分強い" if first_stage.fvalue > 10 else "弱い操作変数")

# 弱い操作変数の例：near_college が教育年数をほとんど動かさないデータ
weak_z = rng_iv.integers(0, 2, n_people)
educ_weak = 12 + 0.1 * weak_z + 1.0 * ability + rng_iv.normal(0, 1.5, n_people)
log_wage_weak = 1.5 + 0.08 * educ_weak + 0.3 * ability + rng_iv.normal(0, 0.3, n_people)
weak = pd.DataFrame({"log_wage": log_wage_weak, "educ": educ_weak, "z": weak_z})

fs_weak = smf.ols("educ ~ z", data=weak).fit()
iv_weak = IV2SLS(weak["log_wage"], sm.add_constant(weak[["educ"]]), instrument=sm.add_constant(weak[["z"]])).fit()
print("弱い操作変数の第 1 段階 F:", round(fs_weak.fvalue, 2))
print("弱い操作変数での 2SLS 推定値:", round(iv_weak.params["educ"], 3), "（標準誤差", round(iv_weak.bse["educ"], 3), "）")

### 6.6 そもそも内生性はあるのか（回帰による Durbin-Wu-Hausman 検定）

「OLS でよいのか、2SLS が必要なのか」を調べる簡単な方法があります。第 1 段階の **残差** を元の回帰式に加え、
その係数が有意なら内生性あり（2SLS を使うべき）と判断します。

In [ ]:
iv_data["v_hat"] = first_stage.resid                       # 第 1 段階の残差
dwh = smf.ols("log_wage ~ educ + v_hat", data=iv_data).fit()
print("残差 v_hat の係数:", round(dwh.params["v_hat"], 3), " p 値:", round(dwh.pvalues["v_hat"], 4))
print("→", "内生性あり：2SLS を使う" if dwh.pvalues["v_hat"] < 0.05 else "内生性の証拠なし：OLS でよい")

> **ローカル環境で linearmodels を使う場合**
> ```python
> from linearmodels.iv import IV2SLS
> res = IV2SLS(dependent=iv_data["log_wage"], exog=sm.add_constant(iv_data[[]]),
>              endog=iv_data["educ"], instruments=iv_data["near_college"]).fit()
> print(res.first_stage)   # 第 1 段階の診断（F 統計量など）も表示される
> ```

### 練習問題 3

1. `iv_data` に、外生的な説明変数として年齢 `age`（`rng_iv.integers(25, 60, n_people)`）を加え、`log_wage ~ educ + age` を OLS と 2SLS（操作変数は `near_college`、`age` は外生変数として操作変数側にも含める）で推定してください。
2. 操作変数が 2 つある場合を試してください：`iv_data` に `parent_educ = 10 + 0.5 * near_college + rng_iv.normal(0, 2, n_people)` を作り、`near_college` と `parent_educ` の両方を操作変数にして 2SLS を推定してください。
3. 第 1 段階の回帰を 2 の操作変数 2 つで行い、F 統計量を確認してください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
iv_data["age"] = rng_iv.integers(25, 60, n_people)
print("OLS :", round(smf.ols("log_wage ~ educ + age", data=iv_data).fit().params["educ"], 4))
X1 = sm.add_constant(iv_data[["educ", "age"]])
Z1 = sm.add_constant(iv_data[["near_college", "age"]])
print("2SLS:", round(IV2SLS(iv_data["log_wage"], X1, instrument=Z1).fit().params["educ"], 4))

# 2
iv_data["parent_educ"] = 10 + 0.5 * iv_data["near_college"] + rng_iv.normal(0, 2, n_people)
Z2 = sm.add_constant(iv_data[["near_college", "parent_educ"]])
print("2SLS（操作変数 2 つ）:", round(IV2SLS(iv_data["log_wage"], sm.add_constant(iv_data[["educ"]]), instrument=Z2).fit().params["educ"], 4))

# 3
fs2 = smf.ols("educ ~ near_college + parent_educ", data=iv_data).fit()
print("第 1 段階 F:", round(fs2.fvalue, 1))
```

</details>

---
## 7. 差の差（DID）

**差の差（difference-in-differences, DID）** は、政策の影響を受けたグループ（処置群）と受けなかったグループ（対照群）の
「前後の変化の差」を政策効果とみなす方法です。パネルデータの代表的な応用です。

$$
y_{it} = \beta_0 + \beta_1 \text{treat}_i + \beta_2 \text{post}_t + \delta\,(\text{treat}_i \times \text{post}_t) + \varepsilon_{it}
$$

交差項の係数 $\delta$ が政策効果です。前提は **平行トレンド仮定**（政策がなければ両群は同じように推移したはず）です。

ここでは、2020 年に一部の企業（処置群）だけが補助金を受け、売上が 0.3 増えたというデータを作ります。

In [ ]:
rng_did = np.random.default_rng(5)
treated_firms = rng_did.choice(np.arange(1, n_firms + 1), size=n_firms // 2, replace=False)

did = panel[["firm", "year", "ad", "size"]].copy()
did["treat"] = did["firm"].isin(treated_firms).astype(int)
did["post"] = (did["year"] >= 2020).astype(int)
did["sales"] = (
    1.0 + 0.5 * did["ad"] + 0.3 * did["size"] + alpha[did["firm"] - 1] + lam[did["year"] - 2016]
    + 0.3 * did["treat"] * did["post"]                       # 政策効果（真の値 0.3）
    + rng_did.normal(0, 0.5, len(did))
)
print(did.groupby(["treat", "post"])["sales"].mean().unstack().round(3))

In [ ]:
# 平行トレンドの確認：処置群と対照群の平均売上の推移
trend = did.groupby(["year", "treat"])["sales"].mean().unstack()
plt.figure(figsize=(7, 4))
plt.plot(trend.index, trend[0], marker="o", label="対照群")
plt.plot(trend.index, trend[1], marker="s", label="処置群")
plt.axvline(2019.5, color="gray", linestyle="--", label="政策開始（2020 年）")
plt.xlabel("年")
plt.ylabel("平均売上（対数）")
plt.title("処置群と対照群の売上の推移")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 2×2 の差の差を手で計算
means = did.groupby(["treat", "post"])["sales"].mean()
did_manual = (means[1, 1] - means[1, 0]) - (means[0, 1] - means[0, 0])
print("手計算の DID 推定値:", round(did_manual, 3), "（真の値 0.3）")

# 回帰による DID（基本形）
did_basic = smf.ols("sales ~ treat + post + treat:post", data=did).fit()
print("回帰の DID 推定値  :", round(did_basic.params["treat:post"], 3))

# 二方向固定効果 + クラスター標準誤差による DID（実務の標準形）
did_fe = smf.ols("sales ~ treat:post + ad + size + C(firm) + C(year)", data=did).fit(
    cov_type="cluster", cov_kwds={"groups": did["firm"]}
)
print("二方向 FE の DID 推定値:", round(did_fe.params["treat:post"], 3), "（標準誤差", round(did_fe.bse["treat:post"], 3), "）")

二方向固定効果モデルでは、`treat`（企業固定効果に吸収される）と `post`（年固定効果に吸収される）を単独で入れる必要はなく、
交差項 `treat:post` だけを入れます。

### 練習問題 4

1. 政策効果が年ごとにどう変わるかを見るため、`did` データで 2020〜2023 年それぞれの `treat × (year == t)` ダミーを作り、二方向固定効果モデルで推定してください（イベントスタディの簡易版）。
2. 平行トレンド仮定のチェックとして、政策前（2016〜2019 年）のデータだけで `sales ~ year_trend + treat:year_trend + C(firm)`（`year_trend = year - 2016`）を推定し、`treat:year_trend` が有意でないことを確認してください（`treat` 単独の項は企業固定効果に吸収されるので入れません）。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
did_es = did.copy()
terms = []
for t in [2020, 2021, 2022, 2023]:
    col = f"treat_{t}"
    did_es[col] = did_es["treat"] * (did_es["year"] == t).astype(int)
    terms.append(col)
es = smf.ols("sales ~ " + " + ".join(terms) + " + ad + size + C(firm) + C(year)", data=did_es).fit(
    cov_type="cluster", cov_kwds={"groups": did_es["firm"]}
)
print(es.params[terms].round(3))

# 2
pre = did[did["year"] < 2020].copy()
pre["year_trend"] = pre["year"] - 2016
pt = smf.ols("sales ~ year_trend + treat:year_trend + C(firm)", data=pre).fit()
print("treat:year_trend の係数:", round(pt.params["treat:year_trend"], 3), " p 値:", round(pt.pvalues["treat:year_trend"], 3))
```

</details>

---
## 8. 結果表の作成

論文やレポートでは、複数のモデルを 1 つの表に並べて比べます。`summary_col()` を使うと、
選んだ係数だけを並べた表を作れます（`*` は有意水準を表す星印）。

In [ ]:
table = summary_col(
    [pooled, fe_lsdv, fe_twoway, fe_cluster],
    stars=True,
    model_names=["プーリング", "FE", "二方向FE", "二方向FE(cluster)"],
    regressor_order=["ad", "size"],
    drop_omitted=True,
    info_dict={"N": lambda r: f"{int(r.nobs)}", "R2": lambda r: f"{r.rsquared:.3f}"},
)
print(table)

### 係数プロット

表の代わりに、係数と 95% 信頼区間を図にすると、手法ごとの違いが直感的に伝わります。

In [ ]:
models = {"プーリング": pooled, "between": between_res, "一階差分": fd, "FE": fe_lsdv, "二方向FE(cluster)": fe_cluster, "RE": re_model}
names = {"一階差分": "d_ad"}
coefs, lows, highs = [], [], []
for label, res in models.items():
    key = names.get(label, "ad")
    ci = res.conf_int().loc[key]
    coefs.append(res.params[key]); lows.append(ci.iloc[0]); highs.append(ci.iloc[1])

plt.figure(figsize=(7, 4))
y_pos = np.arange(len(models))
plt.errorbar(coefs, y_pos, xerr=[np.array(coefs) - np.array(lows), np.array(highs) - np.array(coefs)], fmt="o", capsize=4)
plt.axvline(0.5, color="red", linestyle="--", label="真の値 0.5")
plt.yticks(y_pos, list(models.keys()))
plt.xlabel("広告費の係数（95% 信頼区間）")
plt.title("推定方法ごとの広告費の効果")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# テキストファイルとして保存（左のファイルブラウザに現れます）
with open("panel_results.txt", "w", encoding="utf-8") as f:
    f.write(table.as_text())
print("panel_results.txt に保存しました")

---
## 9. まとめ

| 手法 | いつ使うか | statsmodels での書き方 |
|---|---|---|
| プーリング OLS | 個体効果がない（と信じられる）とき | `smf.ols("y ~ x", data)` |
| 固定効果（FE） | 個体効果が説明変数と相関するとき | `smf.ols("y ~ x + C(id)", data)` |
| 二方向固定効果 | 年ごとの共通ショックも除きたいとき | `smf.ols("y ~ x + C(id) + C(year)", data)` |
| クラスター標準誤差 | 同じ個体の誤差が相関するとき（ほぼ常に） | `.fit(cov_type="cluster", cov_kwds={"groups": data["id"]})` |
| 変量効果（RE） | 個体効果が説明変数と無相関のとき | `smf.mixedlm("y ~ x", data, groups=data["id"])` |
| ハウスマン検定 | FE と RE の選択 | 推定値の差から $\chi^2$ 統計量を計算 |
| 操作変数（2SLS） | 説明変数が内生のとき | `IV2SLS(y, X, instrument=Z).fit()` |
| 差の差（DID） | 政策・介入の効果 | `smf.ols("y ~ treat:post + C(id) + C(year)", data)` |

## 次のステップ

- `python/statsmodels/statsmodels_tutorial.ipynb` — 回帰分析の基礎（診断・ロジスティック回帰・時系列）
- `python/pingouin/pingouin_beginner_tutorial.ipynb` — 検定と効果量
- ローカル環境では `pip install linearmodels` で `PanelOLS` / `IV2SLS` が使えます（各章の囲みを参照）

---
## 総合演習：最低賃金の引き上げと雇用

47 都道府県 × 10 年（2014〜2023 年）の架空のパネルデータを作ります。2019 年に半分の県（処置群）で最低賃金が大きく
引き上げられ、雇用率（対数）に −0.02 の効果があったという設定です。県固定効果は県の産業構造（`manufacturing`：製造業比率）と
相関させてあります。

1. プーリング OLS で `employment ~ min_wage_hike + manufacturing + gdp_growth` を推定してください（`min_wage_hike` は処置群 × 2019 年以降のダミー）。処置群は「もともと雇用が強い県」に偏っているので、プーリング OLS は偏るはずです。
2. 県固定効果と年固定効果を入れ、県でクラスター化した標準誤差で同じモデルを推定してください。真の効果 −0.02 に近づいたか確認しましょう。
3. `manufacturing` の係数が固定効果モデルで推定できない（表に出てこない）理由を考えてください（ヒント：時間を通じて一定の変数）。
4. 処置群と対照群の平均雇用率の推移を折れ線グラフで描き、平行トレンドを目視で確認してください。
5. 1 と 2 の結果を `summary_col` で 1 つの表にまとめてください。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください
rng_mw = np.random.default_rng(99)
n_pref, n_yr = 47, 10
pref = np.repeat(np.arange(1, n_pref + 1), n_yr)
yr = np.tile(np.arange(2014, 2014 + n_yr), n_pref)
manufacturing = np.repeat(rng_mw.uniform(0.1, 0.4, n_pref), n_yr)          # 県ごとに一定
unobserved = rng_mw.normal(0, 0.05, n_pref)                                  # 観測できない県の特性
pref_effect = np.repeat(unobserved, n_yr) + 0.3 * (manufacturing - 0.25)
year_effect = np.tile(np.linspace(0, 0.03, n_yr), n_pref)
treated_pref = np.argsort(unobserved)[-23:] + 1                              # 雇用が強い県ほど引き上げに踏み切った、という設定
treat = np.isin(pref, treated_pref).astype(int)
post = (yr >= 2019).astype(int)
gdp_growth = rng_mw.normal(1.0, 0.5, len(pref))
employment = (
    4.0 + 0.01 * gdp_growth + pref_effect + year_effect - 0.02 * treat * post + rng_mw.normal(0, 0.01, len(pref))
)
mw = pd.DataFrame({
    "pref": pref, "year": yr, "employment": employment, "manufacturing": manufacturing,
    "gdp_growth": gdp_growth, "treat": treat, "post": post,
})
mw["min_wage_hike"] = mw["treat"] * mw["post"]
print(mw.head())

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
# 1. プーリング OLS
mw_pooled = smf.ols("employment ~ min_wage_hike + manufacturing + gdp_growth", data=mw).fit()
print("プーリング OLS の効果:", round(mw_pooled.params["min_wage_hike"], 4))

# 2. 二方向固定効果 + クラスター標準誤差
mw_fe = smf.ols("employment ~ min_wage_hike + gdp_growth + C(pref) + C(year)", data=mw).fit(
    cov_type="cluster", cov_kwds={"groups": mw["pref"]}
)
print("二方向 FE の効果     :", round(mw_fe.params["min_wage_hike"], 4), "（真の値 -0.02）",
      " 標準誤差:", round(mw_fe.bse["min_wage_hike"], 4))

# 3. manufacturing は県ごとに一定なので、県固定効果と完全に重なり（多重共線性）、係数を識別できない。
#    固定効果モデルでは「時間を通じて一定の変数」の効果は推定できず、県固定効果に吸収される。

# 4. 平行トレンド
trend_mw = mw.groupby(["year", "treat"])["employment"].mean().unstack()
plt.figure(figsize=(7, 4))
plt.plot(trend_mw.index, trend_mw[0], marker="o", label="対照群")
plt.plot(trend_mw.index, trend_mw[1], marker="s", label="処置群")
plt.axvline(2018.5, color="gray", linestyle="--", label="引き上げ（2019 年）")
plt.xlabel("年")
plt.ylabel("平均雇用率（対数）")
plt.title("最低賃金引き上げ前後の雇用率")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 5. 結果表
print(summary_col([mw_pooled, mw_fe], stars=True, model_names=["プーリング", "二方向FE"],
                  regressor_order=["min_wage_hike", "gdp_growth", "manufacturing"], drop_omitted=True,
                  info_dict={"N": lambda r: f"{int(r.nobs)}"}))

お疲れさまでした！ パネルデータ分析の核心は「観測できない個体効果をどう扱うか」です。
固定効果で消す・変量効果として扱う・操作変数で内生性に対処する、という 3 つの道具を、
データの性質（個体効果と説明変数の相関、内生性の有無）に応じて使い分けてください。